In [ ]:
from openai import OpenAI

BASE_URL = "http://gpustack.cds.utsa.edu/v1"
API_KEY = "REPLACE-WITH-YOUR-KEY"

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

MODEL = client.models.list().data[1].id 
print("Connected. Model:", MODEL)

Connected. Model: qwen3-vl-8b-instruct


In [5]:
import time
def ask_model(prompt: str, temperature: float = 1.0, retries: int = 2) -> str:
    """Send one prompt to Qwen Coder and return its text reply, retrying on failure."""
    for attempt in range(retries + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
                top_p=0.95,
            )
            return resp.choices[0].message.content
        except Exception:
            if attempt == retries:
                raise
            time.sleep(1.5 * (attempt + 1))
print(ask_model("Say hello in one short sentence."))

Hello!


In [6]:
question = (
    "I am writing a Python function triage_log(line) that asks an LLM to classify a "
    "security log line and returns a dataclass with label and reason. Suggest a robust "
    "prompt format and a defensive way to parse the reply. Keep it concise."
)
print(ask_model(question))

```python
from dataclasses import dataclass
from typing import Optional

@dataclass
class TriageResult:
    label: str
    reason: str

def triage_log(line: str) -> TriageResult:
    prompt = f"""
Classify this log line as one of: 'MALICIOUS', 'BENIGN', 'UNKNOWN'.
Return ONLY: label='...'; reason='...'
No extra text, no markdown.

Line: {line}
"""
    response = ask_llm(prompt)  # your LLM call
    try:
        # Defensive parsing
        if not response.strip():
            return TriageResult(label='UNKNOWN', reason='Empty response')
        if not response.startswith('label='):
            return TriageResult(label='UNKNOWN', reason='Invalid format')
        parts = response.split(';')
        label = parts[0].replace('label=', '').strip().upper()
        reason = parts[1].replace('reason=', '').strip() if len(parts) > 1 else ''
        if not label or label not in {'MALICIOUS', 'BENIGN', 'UNKNOWN'}:
            label = 'UNKNOWN'
        return TriageResult(label=label, reason=reaso

In [7]:
from dataclasses import dataclass
from enum import Enum
import re
class Label(str, Enum):
    BENIGN = "BENIGN"
    SUSPICIOUS = "SUSPICIOUS"
    MALICIOUS = "MALICIOUS"
    UNKNOWN = "UNKNOWN"
@dataclass
class Triage:
    line: str
    label: Label
    reason: str
LABEL_RE = re.compile(r"^LABEL:\s*(\w+)", re.IGNORECASE | re.MULTILINE)
REASON_RE = re.compile(r"^REASON:\s*(.+)", re.IGNORECASE | re.MULTILINE)

In [11]:
def triage_log(line: str) -> Triage:
    prompt = (
        "You are a security log triage assistant.\n"
        "Classify the log line as BENIGN, SUSPICIOUS, or MALICIOUS and give a one "
        "sentence reason.\n"
        "Reply in EXACTLY this format:\n"
        "LABEL: <BENIGN|SUSPICIOUS|MALICIOUS>\n"
        "REASON: <one sentence>\n\n"
        f"Log line:\n{line}\n"
    )
    text = ask_model(prompt)
    
    label_match = LABEL_RE.search(text)
    raw = label_match.group(1).upper() if label_match else "UNKNOWN"
    label = Label(raw) if raw in Label.__members__ else Label.UNKNOWN
    
    reason_match = REASON_RE.search(text)
    reason = reason_match.group(1).strip() if reason_match else text.strip()
    
    return Triage(line=line, label=label, reason=reason)

In [12]:
TOOL_NAME = "SecureAdvisor"

In [13]:
print(triage_log("Failed password for root from 203.0.113.9 port 4022 ssh2"))

Triage(line='Failed password for root from 203.0.113.9 port 4022 ssh2', label=<Label.SUSPICIOUS: 'SUSPICIOUS'>, reason='Multiple failed login attempts for the root account from an external IP indicate potential brute force attack.')


In [14]:
LOG_FILE = "security_log.csv" # Windows CSV, or "security_log.txt" on macOS
MAX_ENTRIES = 20 # keep this first version small and quick
def read_log_lines(path, limit=MAX_ENTRIES):
    with open(path, encoding="utf-8", errors="ignore") as f:
        lines = [ln.strip() for ln in f if ln.strip()]
    return lines[:limit]

entries = read_log_lines(LOG_FILE)
results = [triage_log(line) for line in entries]
print(f"Assessed {len(results)} entries from {LOG_FILE}")

Assessed 20 entries from security_log.csv


In [15]:
from datetime import datetime

def write_report(results, tool_name, path="assessment.md"):
    with open(path, "w", encoding="utf-8") as f:
        f.write(f"# {tool_name} Security Log Assessment\n\n")
        f.write(f"Generated: {datetime.now():%Y-%m-%d %H:%M}\n\n")
        f.write(f"Entries assessed: {len(results)}\n\n")
        for r in results:
            f.write(f"- **{r.label.value}** {r.reason}\n")
            f.write(f"  - `{r.line}`\n")
    return path

report_path = write_report(results, TOOL_NAME)
print("Wrote", report_path, "\n")
print(open(report_path, encoding="utf-8").read())

Wrote assessment.md 

# SecureAdvisor Security Log Assessment

Generated: 2026-11-21 11:07

Entries assessed: 20

- **BENIGN** The log line appears to be a header row for a log file and contains no actual event data to analyze.
  - `﻿Keywords,Date and Time,Source,Event ID,Task Category`
- **BENIGN** This log entry records a successful special logon event, which is a normal audit trail for privilege assignment and does not indicate malicious intent.
  - `Audit Success,11/21/2026 10:55:26 AM,Microsoft-Windows-Security-Auditing,4672,Special Logon,"Special privileges assigned to new logon.`
- **BENIGN** The log line contains no actionable data or security indicators, making it non-threatening.
  - `Subject:`
- **BENIGN** The log entry originates from SYSTEM, which is a standard system process and not indicative of malicious or suspicious activity.
  - `Security ID:		SYSTEM`
- **BENIGN** SYSTEM account is a legitimate built-in account used for system processes and not indicative of maliciou

1. Enum limits the labels to a fixed set of labels (BENIGN, SUSPICIOUS, MALICIOUS, UNKNOWN). Which is why there is no risk that a typing error or some random labeling issue could go through without being checked. Using a dataclass also provides a result with properly named and typed fields (line, label, reason) instead of using a dictionary accepting any key and value without checking.

2. In case the LABEL: entry is not found or does not match any of the set values, the label is set to Label.UNKNOWN instead of having an error that would crash the program. If the REASON: entry is not found or present, the entire reply will be used as the reason.

3. Privilege Lines like SeDebugPrivilege and SeTcbPrivilege as well as Logon ID 0x3E7 are marked as suspicious, but these are the normal privileges and session IDs that Windows grants a SYSTEM account during any special logon, so these should be considered as benign. Also, the SeSecurityPrivilege is marked as BENIGN even though the two entries that appear right before and after this that are in the same privilege line (SeTcbPrivilege and SeTakeOwnershipPrivilege) are SUSPICIOUS, which seems a bit inconsistent in terms of labeling. I would be able to catch issues like this because I can do a spot check for the SUSPICIOUS labels against the security.csv log where I'd see that the tool is judging each line of one log entry on its own, as if every line were a separate event, so it never treats it as a whole. To catch this I would group the lines that belong together into one complete entry before sending them to the model.